# ScienceQA — Inference Notebook

Loads a saved LoRA adapter and generates `submission.csv`.
Run this after the training notebook has saved an adapter.

**Adapter path:** set `ADAPTER_DIR` below to wherever the adapter was saved.
- Same Kaggle session: `./lora-final-seed42`
- Saved as Kaggle dataset output and re-attached: `/kaggle/input/<your-dataset>/lora-final-seed42`

In [ ]:
# ── 0. Install ────────────────────────────────────────────────────────────────
!pip install -q transformers==4.47.1 peft==0.14.0 bitsandbytes accelerate pillow

In [ ]:
# ── 1. Imports & Configuration ────────────────────────────────────────────────
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from tqdm.auto import tqdm
from transformers import Idefics3Processor, Idefics3ForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR    = Path("/kaggle/input/datasets/komvopoulos/finalexamdataset")
ADAPTER_DIR = "./lora-final-seed42"

# ── Model knobs (must match training notebook) ────────────────────────────────
MODEL_ID       = "HuggingFaceTB/SmolVLM-500M-Instruct"
# IMG_SIZE = processor's `longest_edge` cap. 500M variant uses 512×512 patches
# with 64 tokens each (NOT 384). longest_edge=1024 → up to 5 patches/image
# (~320 visual tokens) instead of the ~128 we got at 384.
IMG_SIZE       = 1024
CHOICE_LETTERS = "ABCDEFGHIJ"

# ── Text-field limits (must match training notebook) ─────────────────────────
INCLUDE_LECTURE    = True
LECTURE_MAX_CHARS  = 1000
SOLUTION_MAX_CHARS = 400

# ── System prompt (must match training notebook exactly — simpler version) ───
SYSTEM_PROMPT = (
    "You are an expert science educator. "
    "Carefully analyze the image and all provided context, "
    "then select the single best answer from the choices."
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 2. Load CSVs ──────────────────────────────────────────────────────────────
val_df  = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

for df in [val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

TEST_HAS_SOLUTION = (
    "solution" in test_df.columns
    and test_df["solution"].notna().any()
    and test_df["solution"].astype(str).str.strip().ne("").any()
)
print(f"Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Solution available in test.csv: {TEST_HAS_SOLUTION}")

In [ ]:
# ── 2b. Locate test images ────────────────────────────────────────────────────
# Prints directory layout and probes candidate paths for the first test image.
# Read the output to find the correct root, then set IMAGE_ROOT below.
import os

print("── DATA_DIR contents ──────────────────────────────────────")
for entry in sorted(os.listdir(DATA_DIR)):
    p = DATA_DIR / entry
    tag = "DIR" if p.is_dir() else "file"
    print(f"  [{tag}]  {entry}")

images_dir = DATA_DIR / "images"
if images_dir.exists():
    print("\n── DATA_DIR/images/ contents ──────────────────────────────")
    for entry in sorted(os.listdir(images_dir)):
        p = images_dir / entry
        tag = "DIR" if p.is_dir() else "file"
        print(f"  [{tag}]  {entry}")

sample_path = test_df["image_path"].iloc[0]
print(f"\n── First test image_path value: '{sample_path}' ──────────")
candidates = [
    DATA_DIR / "images" / sample_path,
    DATA_DIR / sample_path,
    DATA_DIR / "images" / Path(sample_path).name,
    DATA_DIR / Path(sample_path).name,
]
for c in candidates:
    print(f"  {'EXISTS ✓' if c.exists() else 'missing  '}: {c}")

# ── Set IMAGE_ROOT to whichever prefix makes paths resolve ───────────────────
# After reading the output above, set this to one of:
#   DATA_DIR / "images"   (nested — worked for train/val)
#   DATA_DIR              (flat — try if nested is missing)
# The _open_image function below uses it automatically.
IMAGE_ROOT = DATA_DIR / "images"   # ← adjust if needed based on output above
print(f"\nIMAGE_ROOT set to: {IMAGE_ROOT}")

In [ ]:
# ── 3. Prompt builder (must match training notebook EXACTLY) ──────────────────
# Reverted to the simpler structure (single Question:, merged Context:) that
# paired with the 80% adapter — the 4-change prompt didn't improve over it.

def _trunc(text: str, max_chars: int) -> str:
    text = str(text).strip()
    return text if len(text) <= max_chars else text[:max_chars] + "…"


def _get(row, key: str, default=""):
    if isinstance(row, dict):
        val = row.get(key, default)
    else:
        val = getattr(row, key, default)
    return default if val is None or (isinstance(val, float) and val != val) else val


def _build_user_text(row, include_solution: bool) -> str:
    """Build the text inside the user message (after the <image>).
    SmolVLM has no native system role, so SYSTEM_PROMPT is prepended here."""
    parts = [SYSTEM_PROMPT]

    # Metadata header
    meta_parts = []
    for key, label in [("subject",  "Subject"),
                       ("topic",    "Topic"),
                       ("category", "Category"),
                       ("grade",    "Grade")]:
        val = str(_get(row, key, "")).strip()
        if val:
            if key == "grade":
                val = val.replace("grade", "Grade ")
            meta_parts.append(f"{label}: {val}")

    skill = str(_get(row, "skill", "")).strip()

    meta_lines = []
    if meta_parts:
        meta_lines.append(" | ".join(meta_parts))
    if skill:
        meta_lines.append(f"Skill: {skill}")
    if meta_lines:
        parts.append("\n".join(meta_lines))

    # Context block (lecture + hint merged together)
    context_parts = []
    if INCLUDE_LECTURE:
        lecture = str(_get(row, "lecture", "")).strip()
        if lecture:
            context_parts.append(_trunc(lecture, LECTURE_MAX_CHARS))

    hint = str(_get(row, "hint", "")).strip()
    if hint:
        context_parts.append(hint)

    if include_solution:
        sol = str(_get(row, "solution", "")).strip()
        if sol:
            context_parts.append("Reasoning: " + _trunc(sol, SOLUTION_MAX_CHARS))

    if context_parts:
        parts.append("Context:\n" + "\n\n".join(context_parts))

    # Question and choices
    choices = _get(row, "choices", [])
    if not isinstance(choices, list):
        choices = []
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )
    question = str(_get(row, "question", "")).strip()

    parts.append(f"Question: {question}\nChoices:\n{choices_str}")

    return "\n\n".join(parts)


def build_prompt(row, include_answer: bool = False, include_solution: bool = False) -> str:
    """
    Output (SmolVLM native template):
      User:<image>{user_text}<end_of_utterance>
      Assistant:                                ← include_answer=False (inference base)
      Assistant: X<end_of_utterance>            ← include_answer=True

    Requires the global `processor` (loaded in cell-inf-load-model).
    """
    proc = globals().get("processor")
    if proc is None:
        raise RuntimeError(
            "processor not loaded — run cell-inf-load-model before scoring"
        )

    user_text = _build_user_text(row, include_solution)

    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": user_text},
    ]}]

    if include_answer:
        letter = CHOICE_LETTERS[int(_get(row, "answer"))]
        messages.append({"role": "assistant", "content": [
            {"type": "text", "text": letter},
        ]})

    prompt = proc.apply_chat_template(
        messages,
        add_generation_prompt=not include_answer,
    )

    # Strip trailing newline so "Assistant:" sits at the very end of the
    # inference prompt — score_choices appends " A" / " B" / ... right there.
    return prompt.rstrip("\n")

In [ ]:
# ── 4. Load saved adapter ─────────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load processor from the BASE model (not ADAPTER_DIR) — Kaggle dataset uploads
# sometimes drop preprocessor_config.json. The processor was never modified by
# fine-tuning, so loading from the HF repo gives an identical (and complete) one.
processor = Idefics3Processor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "right"

# CRITICAL: must match training. Default for SmolVLM-500M is longest_edge=2048
# (4×512 patches). We use 1024 (2×512) — up to 5 patches × 64 tokens ≈ 320
# visual tokens vs ~128 we got at the old 384 thumbnail.
processor.image_processor.size = {"longest_edge": IMG_SIZE}

# Image splitting on: 4 sub-image patches + 1 global thumbnail.
processor.image_processor.do_image_splitting = True

print(f"image_size       : {processor.image_processor.size}")
print(f"image_splitting  : {processor.image_processor.do_image_splitting}")
print(f"padding_side     : {processor.tokenizer.padding_side}")
print(f"pad_token_id     : {processor.tokenizer.pad_token_id}  ({processor.tokenizer.pad_token!r})")
print(f"eos_token_id     : {processor.tokenizer.eos_token_id}  ({processor.tokenizer.eos_token!r})")

# Pin to single GPU — SmolVLM-500M in 4-bit (~250 MB) fits on one T4.
# device_map="auto" splits across both GPUs (pipeline parallelism) which
# adds inter-GPU transfer overhead with no throughput benefit at batch size 1.
base = Idefics3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda:0",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
print(f"Adapter loaded from {ADAPTER_DIR}")
print(f"Model device: {next(model.parameters()).device}")

In [ ]:
# ── 4b. Chat-template verification ────────────────────────────────────────────
# Confirms inference uses the SAME format the model was trained on.
# Train/inference mismatch here = log-likelihood scoring is meaningless.

print("=" * 75)
print("  Inference prompt format check (must match training)")
print("=" * 75)

# Quick reference: what apply_chat_template produces for inference
ref_msgs = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": "What is shown?"},
]}]
ref_inf = processor.apply_chat_template(ref_msgs, add_generation_prompt=True)
print("\n--- apply_chat_template reference (inference) ---")
print(repr(ref_inf))

# Our build_prompt on a real val row
sample_row = val_df.iloc[0].to_dict()
our_inf = build_prompt(sample_row, include_answer=False, include_solution=TEST_HAS_SOLUTION)
print(f"\n--- Our inference prompt ({len(our_inf)} chars) — last 200 chars ---")
print(our_inf[-200:])

# Format checks (same as training verification)
checks = [
    ("Ends with 'Assistant:' (ready for letter append)",
     our_inf.endswith("Assistant:")),
    ("Uses 'User:' (capitalized, inline)",
     "User:" in our_inf and "<|im_start|>user" not in our_inf),
    ("Uses <end_of_utterance> as message separator",
     "<end_of_utterance>" in our_inf),
    ("<image> placeholder appears INLINE right after 'User:'",
     "User:<image>" in our_inf),
    ("Does NOT use <|im_end|> as message separator (old ChatML)",
     "<|im_end|>" not in our_inf),
    ("Does NOT inject 'system' role tag",
     "<|im_start|>system" not in our_inf),
]

all_ok = True
print()
for label, ok in checks:
    icon = "OK  " if ok else "FAIL"
    print(f"  [{icon}] {label}")
    all_ok = all_ok and ok

print()
if all_ok:
    print("  ALL CHECKS PASSED — inference matches the training format.")
else:
    print("  ONE OR MORE CHECKS FAILED — DO NOT score until build_prompt is fixed.")

In [ ]:
# ── 5. Log-likelihood scoring + Test-Time Augmentation (TTA) ──────────────────
#
# TTA via choice permutation: score each question twice — original choice
# order AND reversed — then average the per-choice log-probs. Mapping is by
# ANSWER CONTENT: choice i in original sits at position (n-1-i) in reversed,
# so averaged score for choice i = (lps_orig[i] + lps_rev[n-1-i]) / 2.
#
# Confirmed +2.29% on val for the previous (80%) adapter.
# Cost: 2× forward passes per question.

USE_TTA = True   # set False to disable TTA and recover single-pass behavior


def _open_image(image_path: str, img_size: int = IMG_SIZE) -> Image.Image:
    """Load an image at NATURAL resolution. The processor handles all resizing
    internally based on its `size` config (longest_edge=IMG_SIZE). No manual
    thumbnail — that was discarding ~8× the visual information."""
    path = IMAGE_ROOT / image_path
    if not path.exists():
        # Placeholder for missing images (small grey square is enough — the
        # processor will size it normally).
        return Image.new("RGB", (img_size, img_size), color=(128, 128, 128))
    return Image.open(path).convert("RGB")


@torch.inference_mode()
def score_choices(model, processor, image: Image.Image, row: dict) -> list:
    """Single-pass: return log P(answer_letter | context) for each choice."""
    choices     = row["choices"]
    n           = len(choices)
    prompt_base = build_prompt(row, include_answer=False,
                               include_solution=TEST_HAS_SOLUTION)

    full_texts = [prompt_base + f" {CHOICE_LETTERS[i]}" for i in range(n)]

    processor.tokenizer.padding_side = "right"
    inputs = processor(
        text=full_texts,
        images=[image] * n,
        return_tensors="pt",
        padding=True,
    )
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v
              for k, v in inputs.items()}

    eos_id    = processor.tokenizer.eos_token_id
    pad_id    = processor.tokenizer.pad_token_id
    im_end_id = processor.tokenizer.convert_tokens_to_ids("<|im_end|>")
    skip_ids  = {eos_id, pad_id, im_end_id}

    logits = model(**inputs).logits  # (n, seq_len, vocab_size)

    log_probs = []
    for i in range(n):
        seq = inputs["input_ids"][i].tolist()

        answer_pos = len(seq) - 1
        while answer_pos >= 0 and seq[answer_pos] in skip_ids:
            answer_pos -= 1

        pred_pos        = answer_pos - 1
        answer_token_id = seq[answer_pos]

        lp = torch.log_softmax(logits[i, pred_pos], dim=-1)[answer_token_id].item()
        log_probs.append(lp)

    return log_probs


@torch.inference_mode()
def score_choices_tta(model, processor, image: Image.Image, row: dict) -> list:
    """TTA: average log-probs across original + reversed choice orderings.
    Returns log-probs aligned to ORIGINAL choice indices."""
    n = len(row["choices"])
    lps_orig = score_choices(model, processor, image, row)

    row_rev = {**row, "choices": list(reversed(row["choices"]))}
    lps_rev = score_choices(model, processor, image, row_rev)

    lps_rev_aligned = [lps_rev[n - 1 - i] for i in range(n)]
    return [(lps_orig[i] + lps_rev_aligned[i]) / 2.0 for i in range(n)]


def predict_one(model, processor, image: Image.Image, row: dict,
                use_tta: bool = USE_TTA) -> int:
    score_fn = score_choices_tta if use_tta else score_choices
    lps = score_fn(model, processor, image, row)
    return int(torch.tensor(lps).argmax())


def run_inference(model, processor, df: pd.DataFrame,
                  img_size: int = IMG_SIZE,
                  desc: str = "Inference",
                  use_tta: bool = USE_TTA) -> tuple:
    """Predict all rows in df. Returns (list_of_preds, accuracy_or_None)."""
    model.eval()
    score_fn = score_choices_tta if use_tta else score_choices
    preds, correct, total, missing = [], 0, 0, 0

    print(f"=== Pre-flight diagnostic (first 5 samples, TTA={use_tta}) ===")
    _all_equal = True
    for _di in range(min(5, len(df))):
        _dr  = df.iloc[_di].to_dict()
        _img = _open_image(_dr["image_path"], img_size)
        _lps = score_fn(model, processor, _img, _dr)
        _p   = int(torch.tensor(_lps).argmax())
        _gt  = _dr.get("answer", -1)
        _gt_str = f"  gt={CHOICE_LETTERS[int(_gt)]}" if pd.notna(_gt) and int(_gt) >= 0 else ""
        print(f"  [{_di}] n_choices={len(_lps)}  "
              f"lps={[round(x, 3) for x in _lps]}  "
              f"pred={CHOICE_LETTERS[_p]}{_gt_str}")
        if len(set(round(x, 4) for x in _lps)) > 1:
            _all_equal = False
    if _all_equal:
        print("  WARNING: all log_probs identical — scoring function has a bug.")
    else:
        print("  OK: log_probs show variation — scoring function is working.")
    print("=== End diagnostic ===\n")

    bar = tqdm(df.iterrows(), total=len(df), desc=desc, dynamic_ncols=True)
    for _, row in bar:
        img = _open_image(row["image_path"], img_size)
        # Detect placeholder by checking center pixel against the grey value
        if not (IMAGE_ROOT / row["image_path"]).exists():
            missing += 1
        pred = predict_one(model, processor, img, row.to_dict(), use_tta=use_tta)
        preds.append(pred)

        gt = row.get("answer", -1)
        if pd.notna(gt) and int(gt) >= 0:
            correct += (pred == int(gt))
            total   += 1
            bar.set_postfix(acc=f"{correct}/{total} ({correct/total:.3f})")

    if missing:
        print(f"Note: {missing}/{len(df)} rows used grey placeholder (no image file).")
    acc = correct / total if total > 0 else None
    if acc is not None:
        print(f"Final accuracy: {correct}/{total} = {acc:.4f}")
    return preds, acc

In [ ]:
# ── 6. Predict test set with TTA and write submission.csv ─────────────────────
# TTA confirmed +2.29% on val with the previous adapter — applying it directly
# to the new adapter without re-running val (saves ~30 min).

print(f"=== Test inference (TTA = {USE_TTA}) ===")

test_preds, _ = run_inference(
    model, processor, test_df,
    desc="Test", use_tta=USE_TTA,
)

submission = pd.DataFrame({
    "id":     test_df["id"],
    "answer": test_preds,
})
submission.to_csv("submission.csv", index=False)
print(f"submission.csv written — {len(submission)} rows  (TTA={USE_TTA})")
print(submission.head(10))

In [ ]:
# ── 8. Auto-download submission.csv ───────────────────────────────────────────
import base64
from IPython.display import HTML, display

with open("submission.csv", "rb") as f:
    b64 = base64.b64encode(f.read()).decode()

display(HTML(f"""
<script>
(function() {{
    var a = document.createElement('a');
    a.href = 'data:text/csv;base64,{b64}';
    a.download = 'submission.csv';
    document.body.appendChild(a);
    a.click();
    document.body.removeChild(a);
}})();
</script>
<p>Download triggered. If nothing happened, <a href="data:text/csv;base64,{b64}" download="submission.csv">click here</a>.</p>
"""))